In [1]:
!pip install biopython scikit-learn pandas numpy matplotlib seaborn -q
print("All packages installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.4 MB/s eta 0:00:00
All packages installed successfully.


In [2]:
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from Bio import Entrez, SeqIO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
ENTREZ_EMAIL  = "bilalzaib.microbio@gmail.com"
RANDOM_SEED   = 42
WINDOW_SIZE   = 50
MAX_KMERS     = 256
N_TREES       = 200

In [6]:

DOMAINS = {
    'TAD1':      (1,   40),
    'TAD2':      (41,  67),
    'Proline':   (68, 101),
    'DBD':      (102, 292),
    'Linker':   (293, 324),
    'Tetra':    (325, 356),
    'Regulatory':(357, 393),
}


In [7]:
HOTSPOT_CODONS = {175, 245, 248, 249, 273, 282}

print("Settings ready.")

Settings ready.


In [8]:
def get_domain(codon_number):
    """Returns which TP53 domain a codon belongs to."""
    for domain_name, (start, end) in DOMAINS.items():
        if start <= codon_number <= end:
            return domain_name
    return 'Other'

def get_first_codon(protein_change):
    """
    Extracts the codon number from a protein change like 'R175H, G245S'.
    Takes only the first one when multiple are listed.
    """
    first_entry = str(protein_change).split(',')[0].strip()
    match = re.search(r'(\d+)', first_entry)
    return int(match.group(1)) if match else None

def parse_spdi(spdi_string):
    """
    Parses the Canonical SPDI format: NC_000017.11:7579816:G:C
    Returns position, reference base, alternate base.
    """
    try:
        parts = str(spdi_string).split(':')
        position  = int(parts[1])
        ref_base  = parts[2]
        alt_base  = parts[3]
        return position, ref_base, alt_base
    except:
        return None, None, None

def is_transition(ref_base, alt_base):
    """
    Checks if the base change is a transition (A<->G or C<->T).
    Returns 1 for transition, 0 for transversion, -1 if unknown.
    """
    if ref_base is None:
        return -1
    transitions = {('A','G'), ('G','A'), ('C','T'), ('T','C')}
    return 1 if (ref_base, alt_base) in transitions else 0

def assign_label(clinical_significance):
    """
    Converts ClinVar classification to a number:
    1 = Pathogenic or Likely Pathogenic (harmful)
    0 = Benign or Likely Benign (harmless)
   -1 = Uncertain or Conflicting (excluded)
    """
    sig = str(clinical_significance).lower()
    is_pathogenic = 'pathogenic' in sig
    is_benign     = 'benign'     in sig
    is_conflict   = 'conflicting' in sig

    if is_pathogenic and not is_benign and not is_conflict:
        return 1
    if is_benign and not is_pathogenic and not is_conflict:
        return 0
    return -1  # exclude uncertain/conflicting

def extract_window(cds_sequence, codon_number, window=50):
    """
    Cuts out a section of DNA around the mutation.
    Uses the codon number to find the position in the CDS.
    """
    position_in_cds = (codon_number - 1) * 3
    start = max(0, position_in_cds - window)
    end   = min(len(cds_sequence), position_in_cds + window + 1)
    return cds_sequence[start:end]

# TP53 gene boundaries on chromosome 17 (GRCh38)
TP53_START = 7661779
TP53_END   = 7687538

def genomic_to_relative(position):
    """Converts a genomic coordinate to a value between 0 and 1."""
    return (position - TP53_START) / (TP53_END - TP53_START)
    print("Helper functions ready.")
